# IST DBM 2025 Project - Omarion Aubert, Franciszek Dobrowolski
## Domain

We've decided to design a simple data base to answer questions related to trends in gastronomy:

## Questions in natural language

1. What is the favorite dish for males?
2. Which city has the most seniors eating?
3. What’s the most popular franchise in Louisiana, LA?
4. For every customer, find a restaurant in which they spend the most money in one day.
5. Find the restaurants, in which customers and employees have a different favourite dish.
6. List the restaurants, which had the most customers per month last year.

## ER diagram and relational schema

<img src="diagrams/ER_diagram_Aubert_Dobrowolski.png">

<img src="diagrams/Realtional_schema_Aubert_Dobrowolski.png">

## Connecting to Postgres, creating a data base and connecting to it

In [4]:
#installing libraries if necessary
! pip install psycopg2
! pip install pandas

import psycopg2
import pandas as pd
import pandas.io.sql as psql

from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT

# Connecting to the postgreSQL
try:
    con = psycopg2.connect(user = "postgres",
                                  password = "postgres",
                                  host = "postgres",
                                  port = "5432")
    
    con.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT);
    print("Connected Successfully to PostgreSQL server!!")
    
    cursor = con.cursor();
except (Exception, psycopg2.Error) as error :
     print ("Error while connecting to PostgreSQL", error)

Error while connecting to PostgreSQL could not translate host name "postgres" to address: Name or service not known



In [5]:
# Creating a DB
name_Database   = "Gastronomy";
sqlCreateDatabase = "CREATE DATABASE "+name_Database+";"

try:
    cursor.execute(sqlCreateDatabase);
    print("Database '"+name_Database+"' Created Successfully!")
except (Exception, psycopg2.Error) as error :
    print("Error While Creating the DB: ",error)
    
finally:
    cursor.close()
    con.close()

# get a new connection for the new DB.
con = psycopg2.connect(user = "postgres",
                       password = "postgres",
                       host = "postgres",
                       port = "5432",
                       database = "Gastronomy")

try:
    cursor = con.cursor();
    print("connected again to the server and cusor now on Gastronomy DB !!")
except (Exception, psycopg2.Error) as error:
    print("Error in Connection",error)

Error While Creating the DB:  name 'cursor' is not defined


NameError: name 'cursor' is not defined

## Creating the tables

In [ ]:
#creating customer table
try:
    customerTable="customer"
    create_customerTablee_query = '''CREATE TABLE '''+ customerTable+''' 
    (
        id INT PRIMARY KEY NOT NULL,
        name TEXT NOT NULL,
        age INT NOT NULL,
        gender TEXT
    );'''
    cursor.execute(create_customerTablee_query)
    con.commit()
    print("Table ("+ customerTable +") created successfully in PostgreSQL ")
except (Exception, psycopg2.Error) as error:
    con.rollback()
    print("Error While Creating the DB: ",error)

#creating city table
try:
    cityTable="city"
    create_cityTable_query = '''CREATE TABLE '''+ cityTable+''' 
    (
        id INT PRIMARY KEY NOT NULL,
        name TEXT NOT NULL,
        state TEXT NOT NULL
    ); '''
    cursor.execute(create_cityTable_query)
    con.commit()
    print("Table ("+ cityTable +") created successfully in PostgreSQL ")
except (Exception, psycopg2.Error) as error:
    con.rollback()
    print("Error While Creating the DB: ",error)

#creating restaurant table
try:
    restaurantTable="restaurant"
    create_restaurantTable_query = '''CREATE TABLE '''+ restaurantTable+''' 
    (
        id INT PRIMARY KEY NOT NULL,
        name TEXT NOT NULL,
        city_id INT NOT NULL,
        franchise TEXT,
        FOREIGN KEY (city_id) REFERENCES city(id)
    ); '''
    cursor.execute(create_restaurantTable_query)
    con.commit()
    print("Table ("+ restaurantTable +") created successfully in PostgreSQL ")
except (Exception, psycopg2.Error) as error:
    con.rollback()
    print("Error While Creating the DB: ",error)

#creating food table
try:
    foodTable="food"
    create_foodTable_query = '''CREATE TABLE '''+ foodTable+''' 
    (
        id INT PRIMARY KEY NOT NULL,
        name TEXT NOT NULL,
        price INT NOT NULL
    ); '''
    cursor.execute(create_foodTable_query)
    con.commit()
    print("Table ("+ foodTable +") created successfully in PostgreSQL ")
except (Exception, psycopg2.Error) as error:
    con.rollback()
    print("Error While Creating the DB: ",error)

#creating employee table
try:
    employeeTable="employee"
    create_employeeTable_query = '''CREATE TABLE '''+ employeeTable+''' 
    (
        id INT PRIMARY KEY NOT NULL,
        name TEXT NOT NULL,
        restaurant_id INT NOT NULL,
        favourite_food_id INT NOT NULL,
        FOREIGN KEY (restaurant_id) REFERENCES restaurant(id),
        FOREIGN KEY (favourite_food_id) REFERENCES food(id)
    ); '''
    cursor.execute(create_employeeTable_query)
    con.commit()
    print("Table ("+ employeeTable +") created successfully in PostgreSQL ")
except (Exception, psycopg2.Error) as error:
    con.rollback()
    print("Error While Creating the DB: ",error)

#creating orders table
try:
    orderTable="orders"
    create_orderTable_query = '''CREATE TABLE '''+ orderTable+''' 
    (
        id INT PRIMARY KEY NOT NULL,
        customer_id INT NOT NULL,
        restaurant_id INT NOT NULL,
        food_id INT NOT NULL,
        order_date DATE NOT NULL,
        FOREIGN KEY (customer_id) REFERENCES customer(id),
        FOREIGN KEY (restaurant_id) REFERENCES restaurant(id),
        FOREIGN KEY (food_id) REFERENCES food(id)
    ); '''
    cursor.execute(create_orderTable_query)
    con.commit()
    print("Table ("+ orderTable +") created successfully in PostgreSQL ")
except (Exception, psycopg2.Error) as error:
    con.rollback()
    print("Error While Creating the DB: ",error)

## Filling the tables
The data generated using Mockaroo.

In [ ]:
import csv
from psycopg2 import sql

def load_csv_to_table(cursor, table_name, csv_path):
    with open(csv_path, 'r', encoding='utf-8') as f:
        next(f)  # pominięcie nagłówka
        cursor.copy_expert(
            sql.SQL("COPY {} FROM STDIN WITH CSV").format(sql.Identifier(table_name)),
            f
        )

try:
    load_csv_to_table(cursor, "customer", "customers.csv")
    con.commit()
    print("customer loaded")
    
    load_csv_to_table(cursor, "city", "cities.csv")
    con.commit()
    print("city loaded")
    
    load_csv_to_table(cursor, "restaurant", "restaurants.csv")
    con.commit()
    print("restaurant loaded")
    
    load_csv_to_table(cursor, "food", "foods.csv")
    con.commit()
    print("food loaded")
    
    load_csv_to_table(cursor, "employee", "employees.csv")
    con.commit()
    print("employee loaded")
    
    load_csv_to_table(cursor, "orders", "orders.csv")
    con.commit()
    print("orders loaded")

except Exception as e:
    con.rollback()
    print("Error:", e)


## Queries

### Query nr 1
What is the favorite dish for men?

π{f.name} ( σ{c.gender='male'} ( (order ⋈{o.customer_id=c.id} customer) ⋈{o.food_id=f.id} food ) )


In [ ]:
query1 = psql.read_sql("""
    SELECT f.name AS favorite_dish, COUNT(*) AS orders_count
    FROM orders o
    JOIN customer c ON o.customer_id = c.id
    JOIN food f ON o.food_id = f.id
    WHERE c.gender = 'male'
    GROUP BY f.name
    ORDER BY orders_count DESC
    LIMIT 3;
""", con)
display(query1)

### Query nr 2
Which city has the most seniors eating?

π{ci.name, o.customer_id} ( σ{c.age >= 65} ( (((order ⋈{o.customer_id=c.id} customer) ⋈{o.restaurant_id=r.id} restaurant) ⋈{r.city_id=ci.id} city) ) )


In [ ]:
query2 = psql.read_sql("""
    SELECT ci.name AS city_name, COUNT(DISTINCT o.customer_id) AS senior_count
    FROM orders o
    JOIN customer c ON o.customer_id = c.id
    JOIN restaurant r ON o.restaurant_id = r.id
    JOIN city ci ON r.city_id = ci.id
    WHERE c.age >= 65
    GROUP BY ci.name
    ORDER BY senior_count DESC
    LIMIT 3;
""", con)
display(query2)

### Query nr 3
What’s the most popular franchise in Louisiana, LA?

π{r.franchise, o.id} ( σ{c.state = 'LA'} ( ( (order ⋈{o.restaurant_id = r.id} restaurant ) ⋈{r.city_id = c.id} city ) ) )

In [ ]:
query3 = psql.read_sql("""
    SELECT r.franchise, COUNT(*) AS orders_count
    FROM orders o
    JOIN restaurant r ON o.restaurant_id = r.id
    JOIN city c ON r.city_id = c.id
    WHERE c.state = 'LA'
    GROUP BY r.franchise
    ORDER BY orders_count DESC
    LIMIT 3;
""", con)
display(query3)

### Query nr 4
For every customer, find a restaurant in which they spend the most money in one day.

Simplified, because of sum, groups, etc.

R = (order ⋈{o.customer_id = c.id} customer)
      ⋈{o.food_id = f.id} food
      ⋈{o.restaurant_id = r.id} restaurant

Rfinal = π{c.id, c.name, o.restaurant_id, r.name, o.order_date}(R)


In [ ]:
query4 = psql.read_sql("""
    SELECT
        c.id AS customer_id,
        c.name AS customer_name,
        o.restaurant_id,
        r.name AS restaurant_name,
        o.order_date,
        SUM(f.price) AS total_spent
    FROM
        orders o
    JOIN customer c ON o.customer_id = c.id
    JOIN food f ON o.food_id = f.id
    JOIN restaurant r ON o.restaurant_id = r.id
    GROUP BY c.id, c.name, o.restaurant_id, r.name, o.order_date
    HAVING SUM(f.price) = (
        SELECT MAX(total)
        FROM (
            SELECT SUM(f2.price) AS total
            FROM orders o2
            JOIN food f2 ON o2.food_id = f2.id
            WHERE o2.customer_id = c.id
            GROUP BY o2.restaurant_id, o2.order_date
        ) AS sub
    );
""", con)
display(query4)

### Query nr 5
Find the restaurants, in which customers and employees have a different favourite dish.

Simplified:

CustOrders = order ⋈{o.restaurant_id = r.id} restaurant

EmpFav = employee ⋈{e.restaurant_id = r.id} restaurant


Rdiff = σ{o.food_id != e.favourite_food_id}(CustOrders ⋈{r.id = e.restaurant_id} EmpFav)


Rfinal = π{r.id, r.name}(Rdiff)

In [ ]:
query5 = psql.read_sql("""
    WITH customer_fav AS (
    SELECT
        o.restaurant_id,
        o.food_id,
        COUNT(*) AS cnt
    FROM orders o
    GROUP BY o.restaurant_id, o.food_id
),
customer_top AS (
    SELECT restaurant_id, food_id
    FROM customer_fav cf
    WHERE cf.cnt = (
        SELECT MAX(cf2.cnt)
        FROM customer_fav cf2
        WHERE cf2.restaurant_id = cf.restaurant_id
    )
),
employee_fav AS (
    SELECT
        e.restaurant_id,
        o.food_id,
        COUNT(*) AS cnt
    FROM employee e
    JOIN orders o ON e.restaurant_id = o.restaurant_id
    GROUP BY e.restaurant_id, o.food_id
),
employee_top AS (
    SELECT restaurant_id, food_id
    FROM employee_fav ef
    WHERE ef.cnt = (
        SELECT MAX(ef2.cnt)
        FROM employee_fav ef2
        WHERE ef2.restaurant_id = ef.restaurant_id
    )
)
SELECT DISTINCT r.id, r.name
FROM restaurant r
JOIN customer_top ct ON r.id = ct.restaurant_id
JOIN employee_top et ON r.id = et.restaurant_id
WHERE ct.food_id <> et.food_id;

""", con)
display(query5)

### Query nr 6
List the restaurants, which had the most customers per month last year.

R1 = σ{YEAR(o.order_date) = YEAR(CURRENT_DATE)-1}(order)

R2 = R1 ⋈{o.restaurant_id = r.id} restaurant

Rfinal = π{r.name, DATE_TRUNC('month', o.order_date), o.customer_id}(R2)

In [ ]:
query6 = psql.read_sql("""
    SELECT r.name,
       DATE_TRUNC('month', o.order_date) AS month,
       COUNT(DISTINCT o.customer_id) AS customers_count
    FROM orders o
    JOIN restaurant r ON o.restaurant_id = r.id
    WHERE EXTRACT(YEAR FROM o.order_date) = EXTRACT(YEAR FROM CURRENT_DATE) - 1
    GROUP BY r.name, DATE_TRUNC('month', o.order_date)
    ORDER BY customers_count DESC
    LIMIT 3;
""", con)
display(query6)